# LLM-Based Relation Extraction — Step 2

Extracts semantically-typed edges from paper abstracts using a local Ollama model.  
Both endpoints of every edge must already exist in `unified_kg_nodes`.  
Output: `nlp_kg_edges_llm.parquet` with schema matching `unified_kg_edges`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

# Ensure src is on path when running from experiments/
repo_root = Path("../").resolve()
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

from neurovlm.gnn.llm_re import (
    RELATION_TYPES,
    SYSTEM_PROMPT,
    build_user_prompt,
    call_ollama,
    validate_triple,
    process_paper,
    aggregate_results,
)

DATA = repo_root / "experiments" / "data"

PAPERS_PATH       = DATA / "pubmed_neuroimaging.parquet"
ANNOTATIONS_PATH  = DATA / "mesh_kg"    / "mesh_annotations_long.parquet"
DESCRIPTORS_PATH  = DATA / "mesh_kg"    / "mesh_descriptors.parquet"
KG_NODES_PATH     = DATA / "unified_kg" / "unified_kg_nodes.parquet"
STEP1_EDGES_PATH  = DATA / "nlp_kg"     / "nlp_kg_edges_qualified.parquet"
OUTPUT_PATH       = DATA / "nlp_kg"     / "nlp_kg_edges_llm.parquet"
CHECKPOINT_DIR    = DATA / "nlp_kg"     / "llm_re_cache"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# Model — prefer 14b if available
OLLAMA_MODEL = "qwen2.5:14b-instruct"
PAPER_CAP    = 100_000   # maximum papers to process in the initial run

print(f"Relation types   : {sorted(RELATION_TYPES)}")
print(f"Model            : {OLLAMA_MODEL}")
print(f"Paper cap        : {PAPER_CAP:,}")
print(f"Checkpoint dir   : {CHECKPOINT_DIR}")
print("Paths OK")

Relation types   : ['associated_with_disorder', 'co_activates_with', 'expressed_in', 'implicated_in', 'treated_by', 'used_in']
Model            : qwen2.5:14b-instruct
Paper cap        : 100,000
Checkpoint dir   : /Users/borng/code/lab_work/neurovlm/experiments/data/nlp_kg/llm_re_cache
Paths OK


In [16]:
# Cell 2 — load mesh_descriptors → build name_to_ui
descriptors_df = pd.read_parquet(DESCRIPTORS_PATH)
print(f"Descriptors shape: {descriptors_df.shape}")

# Lowercase name → DescriptorUI (primary lookup)
name_to_ui: dict[str, str] = dict(
    zip(
        descriptors_df["name"].str.lower().str.strip(),
        descriptors_df["ui"],
    )
)

# Also map original-case names (for validate_triple term lookup)
# We keep both because LLM outputs may preserve original MeSH casing.
ui_to_name: dict[str, str] = dict(zip(descriptors_df["ui"], descriptors_df["name"]))

print(f"name_to_ui size  : {len(name_to_ui):,}")
print(f"Sample entries   :")
for k, v in list(name_to_ui.items())[:4]:
    print(f"  '{k}' → {v}")

Descriptors shape: (31110, 4)
name_to_ui size  : 31,110
Sample entries   :
  'calcimycin' → D000001
  'temefos' → D000002
  'abattoirs' → D000003
  'abbreviations as topic' → D000004


In [17]:
# Cell 3 — load unified_kg_nodes → build kg_node_ids, node_type lookup
kg_nodes_df = pd.read_parquet(KG_NODES_PATH)
print(f"KG nodes shape   : {kg_nodes_df.shape}")
print(f"Node types       :\n{kg_nodes_df['node_type'].value_counts().to_string()}")

kg_node_ids: set[str] = set(kg_nodes_df["canonical_id"])
id_to_type: dict[str, str] = dict(
    zip(kg_nodes_df["canonical_id"], kg_nodes_df["node_type"])
)
id_to_name: dict[str, str] = dict(
    zip(kg_nodes_df["canonical_id"], kg_nodes_df["name"])
)
# Augment ui_to_name with non-MeSH KG node names
ui_to_name.update(id_to_name)

# Relevant node types for priority filtering
RICH_TYPES = {"cognitive_construct", "anatomical_region", "disorder", "method", "intervention"}
MOLECULAR_ANAT = {"molecular", "anatomical_region"}

# Build set of KG-backed MeSH names (original case, for fast set membership)
kg_mesh_ui_set: set[str] = {uid for uid in kg_node_ids if uid.startswith("D") or uid.startswith("C")}
print(f"\nKG node IDs total: {len(kg_node_ids):,}")

KG nodes shape   : (33784, 5)
Node types       :
node_type
molecular              10671
disorder                5537
organism                3938
other                   3196
method                  3023
anatomical_region       2518
cognitive_construct     2116
biological_process      1886
intervention             422
demographic              345
network                  132

KG node IDs total: 33,784


In [18]:
# Cell 4 — load annotations → build per-paper MeSH term lists (join with papers)
print("Loading annotations...")
annotations_df = pd.read_parquet(ANNOTATIONS_PATH)
print(f"Annotations shape: {annotations_df.shape}")

# Extract base term (strip qualifier suffix)
annotations_df["base_term"] = annotations_df["mesh_term"].str.split("/").str[0].str.strip()

# Map base term → canonical_id via name_to_ui
annotations_df["canonical_id"] = annotations_df["base_term"].str.lower().map(name_to_ui)

# Keep only terms that map to a KG node
ann_in_kg = annotations_df[
    annotations_df["canonical_id"].notna()
    & annotations_df["canonical_id"].isin(kg_node_ids)
].copy()
print(f"Annotations in KG: {len(ann_in_kg):,}  (of {len(annotations_df):,})")

# Normalize pmid dtype to string for joining with papers parquet
ann_in_kg["pmid"] = ann_in_kg["pmid"].astype(str)

# Per-paper: list of (base_term, canonical_id, node_type)
ann_in_kg["node_type"] = ann_in_kg["canonical_id"].map(id_to_type)

# Group to dict: pmid → list of (base_term, canonical_id, node_type)
paper_terms: dict[str, list[tuple]] = (
    ann_in_kg
    .drop_duplicates(subset=["pmid", "canonical_id"])
    .groupby("pmid")[["base_term", "canonical_id", "node_type"]]
    .apply(lambda g: list(zip(g["base_term"], g["canonical_id"], g["node_type"])))
    .to_dict()
)
print(f"Papers with ≥1 KG-backed MeSH term: {len(paper_terms):,}")
print("Sample (first paper):")
sample_pmid = next(iter(paper_terms))
print(f"  pmid={sample_pmid}: {paper_terms[sample_pmid][:5]}")

Loading annotations...
Annotations shape: (14470827, 2)
Annotations in KG: 14,470,827  (of 14,470,827)
Papers with ≥1 KG-backed MeSH term: 1,032,953
Sample (first paper):
  pmid=10021308: [('Adenocarcinoma', 'D000230', 'disorder'), ('Aged', 'D000368', 'demographic'), ('Chemotherapy, Adjuvant', 'D017024', 'method'), ('Endometrial Neoplasms', 'D016889', 'disorder'), ('Female', 'D005260', 'other')]


In [19]:
# Cell 5 — apply filtering strategy → print counts at each step → get pmid subset
print("Applying paper filtering strategy...\n")

# ---- Priority 1: cognitively rich papers ----
p1_pmids: set[str] = set()
for pmid, terms in paper_terms.items():
    if sum(1 for _, _, nt in terms if nt in RICH_TYPES) >= 2:
        p1_pmids.add(pmid)
print(f"Priority 1 (≥2 cognitively-rich terms)   : {len(p1_pmids):>8,}")

# ---- Priority 2: Step 1 gaps ----
print("Loading Step 1 edges for gap detection...")
step1_df = pd.read_parquet(STEP1_EDGES_PATH)

ann_with_rel_types = set(
    ann_in_kg[
        ann_in_kg["mesh_term"].str.contains("/", na=False)
        & ann_in_kg["mesh_term"]
              .str.split("/").str[1]
              .str.lower()
              .isin([
                  "physiology", "physiopathology", "pathology", "genetics",
                  "metabolism", "psychology", "diagnosis", "diagnostic imaging",
                  "complications", "etiology", "methods", "instrumentation",
                  "drug therapy", "therapeutic use", "therapy",
              ])
    ]["pmid"]
)

p2_pmids: set[str] = set()
for pmid, terms in paper_terms.items():
    if pmid not in ann_with_rel_types and len(terms) >= 2:
        p2_pmids.add(pmid)
print(f"Priority 2 (Step 1 gaps, ≥2 terms)       : {len(p2_pmids):>8,}")

# ---- Priority 3: expressed_in / co_activates_with targets ----
p3_pmids: set[str] = set()
for pmid, terms in paper_terms.items():
    types_present = {nt for _, _, nt in terms}
    if "molecular" in types_present and "anatomical_region" in types_present:
        p3_pmids.add(pmid)
print(f"Priority 3 (molecular ∩ anatomical_region): {len(p3_pmids):>8,}")

# ---- Union and sort by rich-term pair count ----
candidate_pmids = p1_pmids | p2_pmids | p3_pmids
print(f"\nUnion of all priorities                  : {len(candidate_pmids):>8,}")

def n_rich_pairs(pmid: str) -> int:
    n = sum(1 for _, _, nt in paper_terms[pmid] if nt in RICH_TYPES)
    return n * (n - 1) // 2

sorted_pmids = sorted(candidate_pmids, key=n_rich_pairs, reverse=True)

# ---- Load abstracts now so we can filter before capping ----
print("\nLoading papers for abstract text...")
papers_df = pd.read_parquet(PAPERS_PATH, columns=["pmid", "abstract"])
papers_df["pmid"] = papers_df["pmid"].astype(str)
papers_df = papers_df.set_index("pmid")
print(f"Papers loaded: {len(papers_df):,}")

# Drop papers whose abstract is missing or blank — they always produce [] and
# waste a checkpoint slot.
has_abstract: set[str] = {
    pmid for pmid in sorted_pmids
    if pmid in papers_df.index
    and isinstance(papers_df.loc[pmid, "abstract"], str)
    and papers_df.loc[pmid, "abstract"].strip()
}
sorted_pmids = [p for p in sorted_pmids if p in has_abstract]
print(f"After dropping empty-abstract papers     : {len(sorted_pmids):>8,}")

# ---- Exclude already-checkpointed papers so each run advances the window ----
already_checkpointed: set[str] = {cp.stem for cp in CHECKPOINT_DIR.glob("*.json")}
unprocessed_pmids = [p for p in sorted_pmids if p not in already_checkpointed]
print(f"Already checkpointed (prior runs)        : {len(already_checkpointed):>8,}")
print(f"Remaining unprocessed candidates         : {len(unprocessed_pmids):>8,}")

pmid_subset = unprocessed_pmids[:PAPER_CAP]
print(f"This run's batch (cap={PAPER_CAP:,})            : {len(pmid_subset):>8,}")

Applying paper filtering strategy...

Priority 1 (≥2 cognitively-rich terms)   : 1,001,428
Loading Step 1 edges for gap detection...
Priority 2 (Step 1 gaps, ≥2 terms)       :   66,945
Priority 3 (molecular ∩ anatomical_region):  219,788

Union of all priorities                  : 1,017,002

Loading papers for abstract text...
Papers loaded: 1,231,613
After dropping empty-abstract papers     :  927,984
Already checkpointed (prior runs)        :        0
Remaining unprocessed candidates         :  927,984
This run's batch (cap=100,000)            :  100,000


In [20]:
# Cell 6 — load Step 1 edges → build existing_edges frozenset
print(f"Step 1 edges shape: {step1_df.shape}")
existing_edges: frozenset = frozenset(
    zip(step1_df["subject_id"], step1_df["relation_type"], step1_df["object_id"])
)
print(f"Existing edge triples (Step 1): {len(existing_edges):,}")
print("Step 1 relation distribution:")
print(step1_df["relation_type"].value_counts().to_string())

Step 1 edges shape: (14790106, 6)
Existing edge triples (Step 1): 14,790,106
Step 1 relation distribution:
relation_type
implicated_in               6619094
associated_with_disorder    4185358
treated_by                  2695376
used_in                     1290278


In [21]:
# Cell 7 — test single paper: show prompt + raw LLM response + parsed output

# Pick the first paper in the subset that has a non-empty abstract.
# (Cell 5 already filters these out, but this guards against stale kernel state.)
test_pmid = next(
    (p for p in pmid_subset
     if p in papers_df.index
     and isinstance(papers_df.loc[p, "abstract"], str)
     and papers_df.loc[p, "abstract"].strip()),
    pmid_subset[0] if pmid_subset else None,
)
if test_pmid is None:
    raise RuntimeError("pmid_subset is empty — re-run Cell 5.")

test_terms_raw  = paper_terms[test_pmid]
test_mesh_names = [name for name, _, _ in test_terms_raw]
test_abstract   = str(papers_df.loc[test_pmid, "abstract"])

print(f"Test PMID        : {test_pmid}")
print(f"MeSH terms ({len(test_mesh_names)}): {test_mesh_names[:10]}{'...' if len(test_mesh_names) > 10 else ''}")
print(f"Abstract (first 300 chars):\n  {test_abstract[:300]}")
print()

user_prompt = build_user_prompt(test_abstract, test_mesh_names)
print("=== User Prompt ===")
print(user_prompt[:1200])
print("\n=== Calling LLM ===")

raw_triples = call_ollama(user_prompt, model=OLLAMA_MODEL)
print(f"Raw triples returned: {len(raw_triples)}")
for t in raw_triples:
    print(f"  {t}")

print("\n=== Validated Triples ===")
valid_set = set(test_mesh_names)
for t in raw_triples:
    ok = validate_triple(t, valid_set, RELATION_TYPES)
    print(f"  {'✓' if ok else '✗'} {t}")

Test PMID        : 27132696
MeSH terms (47): ['Acute Kidney Injury', 'Anemia', 'Antineoplastic Combined Chemotherapy Protocols', 'Central Nervous System Neoplasms', 'Chemical and Drug Induced Liver Injury', 'Combined Modality Therapy', 'Comparative Effectiveness Research', 'Cytarabine', 'Death, Sudden', 'Denmark']...
Abstract (first 300 chars):
  BACKGROUND: Standard treatment for patients with primary CNS lymphoma remains to be defined. Active therapies are often associated with increased risk of haematological or neurological toxicity. In this trial, we addressed the tolerability and efficacy of adding rituximab with or without thiotepa to

=== User Prompt ===
Abstract:
BACKGROUND: Standard treatment for patients with primary CNS lymphoma remains to be defined. Active therapies are often associated with increased risk of haematological or neurological toxicity. In this trial, we addressed the tolerability and efficacy of adding rituximab with or without thiotepa to methotrexate-cytar

In [23]:
import json
# Cell 8 — run full extraction loop with tqdm progress bar (resumable)
already_done = len(list(CHECKPOINT_DIR.glob("*.json")))
print(f"Checkpoints already on disk: {already_done:,}")
print(f"Papers to process          : {len(pmid_subset):,}")
print(f"Model                      : {OLLAMA_MODEL}")
print("Starting extraction loop (Ctrl-C to pause — fully resumable)...\n")

n_empty   = 0   # papers where LLM returned []
n_invalid = 0   # triples rejected by validate_triple
n_total_raw = 0

for pmid in tqdm(pmid_subset, desc="LLM extraction", unit="paper"):
    checkpoint_path = CHECKPOINT_DIR / f"{pmid}.json"

    if checkpoint_path.exists():
        # Already processed — count for stats but skip LLM call
        continue

    terms_raw = paper_terms.get(pmid, [])
    mesh_names = [name for name, _, _ in terms_raw]
    abstract = str(papers_df.loc[pmid, "abstract"]) if pmid in papers_df.index else ""

    if not abstract or not mesh_names:
        checkpoint_path.write_text("[]")
        n_empty += 1
        continue

    # Build prompt and call LLM
    user_prompt = build_user_prompt(abstract, mesh_names)
    raw_triples = call_ollama(user_prompt, model=OLLAMA_MODEL)

    # Validate
    valid_set = set(mesh_names)
    valid_raw = [t for t in raw_triples if validate_triple(t, valid_set, RELATION_TYPES)]

    n_total_raw += len(raw_triples)
    n_invalid   += len(raw_triples) - len(valid_raw)
    if not valid_raw:
        n_empty += 1

    # Write checkpoint
    checkpoint_path.write_text(json.dumps(valid_raw))

print(f"\nExtraction complete.")
print(f"  Papers with no valid triples (LLM returned []) : {n_empty:,}")
print(f"  Raw triples generated                          : {n_total_raw:,}")
print(f"  Triples rejected by validate_triple            : {n_invalid:,}")
if n_total_raw:
    print(f"  Rejection rate                                 : {100*n_invalid/n_total_raw:.1f}%")

Checkpoints already on disk: 0
Papers to process          : 100,000
Model                      : qwen2.5:14b-instruct
Starting extraction loop (Ctrl-C to pause — fully resumable)...



LLM extraction:   0%|          | 0/100000 [00:00<?, ?paper/s]

KeyboardInterrupt: 

In [24]:
import json

# Cell 9 — aggregate checkpoint files → show relation type distribution
print("Aggregating checkpoint files...")
llm_edges_df = aggregate_results(
    checkpoint_dir=CHECKPOINT_DIR,
    name_to_ui=name_to_ui,
    existing_edges=existing_edges,
    kg_node_ids=kg_node_ids,
)
print(f"\nLLM edges shape: {llm_edges_df.shape}")
print("\nRelation type distribution:")
counts = llm_edges_df["relation_type"].value_counts()
for rel, cnt in counts.items():
    print(f"  {rel:<30} {cnt:>10,}  ({100*cnt/len(llm_edges_df):.1f}%)")

print(f"\nWeight statistics:")
print(llm_edges_df["weight"].describe().to_string())

Aggregating checkpoint files...

LLM edges shape: (52, 6)

Relation type distribution:
  used_in                                19  (36.5%)
  expressed_in                           19  (36.5%)
  implicated_in                           7  (13.5%)
  treated_by                              5  (9.6%)
  co_activates_with                       2  (3.8%)

Weight statistics:
count    52.0
mean      1.0
std       0.0
min       1.0
25%       1.0
50%       1.0
75%       1.0
max       1.0


In [25]:
# Cell 10 — spot-check: 5 examples per relation type with human-readable names
sample = llm_edges_df.copy()
sample["subject_name"] = sample["subject_id"].map(ui_to_name)
sample["object_name"]  = sample["object_id"].map(ui_to_name)

for rel_type in sorted(RELATION_TYPES):
    group = sample[sample["relation_type"] == rel_type]
    if group.empty:
        print(f"\n── {rel_type}: NO EDGES FOUND ──")
        continue
    print(f"\n── {rel_type} ({len(group):,} edges — showing top 5 by weight) ──")
    cols = ["subject_name", "relation_type", "object_name", "weight"]
    top5 = group.nlargest(5, "weight")[cols]
    for _, row in top5.iterrows():
        print(f"  {row['subject_name']}  →  {row['relation_type']}  →  {row['object_name']}  (weight={row['weight']:.0f})")


── associated_with_disorder: NO EDGES FOUND ──

── co_activates_with (2 edges — showing top 5 by weight) ──
  Mammillary Bodies  →  co_activates_with  →  Third Ventricle  (weight=1)
  Lymphocyte Activation  →  co_activates_with  →  Interleukin-2  (weight=1)

── expressed_in (19 edges — showing top 5 by weight) ──
  Hamartoma  →  expressed_in  →  Hypothalamus  (weight=1)
  Astrocytoma  →  expressed_in  →  Cerebellum  (weight=1)
  Oval Window, Ear  →  expressed_in  →  Ear, Middle  (weight=1)
  Ethmoid Bone  →  expressed_in  →  Cranial Fossa, Anterior  (weight=1)
  Sphenoid Bone  →  expressed_in  →  Cranial Fossa, Middle  (weight=1)

── implicated_in (7 edges — showing top 5 by weight) ──
  Trastuzumab  →  implicated_in  →  Primary Prevention  (weight=1)
  Diffusion Magnetic Resonance Imaging  →  implicated_in  →  Diagnosis, Differential  (weight=1)
  Magnetic Resonance Spectroscopy  →  implicated_in  →  Diagnosis, Differential  (weight=1)
  Watchful Waiting  →  implicated_in  →  Clinica

In [29]:
# Cell 11 — validation stats (rejection rate, invalid triple rate, coverage)
n_checkpoints = len(list(CHECKPOINT_DIR.glob("*.json")))
n_empty_checkpoints = 0
n_raw_total = 0

for cp in CHECKPOINT_DIR.glob("*.json"):
    try:
        triples = json.loads(cp.read_text())
        n_raw_total += len(triples)
        if not triples:
            n_empty_checkpoints += 1
    except (json.JSONDecodeError, OSError):
        pass

rejection_rate = 100 * n_empty_checkpoints / n_checkpoints if n_checkpoints else 0.0

print("=== Validation Statistics ===")
print(f"Checkpoints written          : {n_checkpoints:>10,}")
print(f"  Papers with [] result      : {n_empty_checkpoints:>10,}  ({rejection_rate:.1f}% rejection)")
print(f"  Papers with ≥1 triple      : {n_checkpoints - n_empty_checkpoints:>10,}")
print(f"Validated triples in cache   : {n_raw_total:>10,}")
print(f"Final aggregated edges       : {len(llm_edges_df):>10,}")
print(f"Edges deduped vs Step 1      : {len(llm_edges_df):>10,}  (already excluded)")

# Coverage: unique node pairs added vs Step 1
step1_pairs = len(existing_edges)
llm_pairs = len(llm_edges_df)
step1_unique_nodes = len(
    set(step1_df["subject_id"]) | set(step1_df["object_id"])
)
llm_unique_nodes = len(
    set(llm_edges_df["subject_id"]) | set(llm_edges_df["object_id"])
) if not llm_edges_df.empty else 0
print(f"\nStep 1 unique node pairs     : {step1_pairs:>10,}")
print(f"Step 2 new unique node pairs : {llm_pairs:>10,}")
print(f"Step 1 unique nodes involved : {step1_unique_nodes:>10,}")
print(f"Step 2 unique nodes involved : {llm_unique_nodes:>10,}")

=== Validation Statistics ===
Checkpoints written          :     43,784
  Papers with [] result      :     43,706  (99.8% rejection)
  Papers with ≥1 triple      :         78
Validated triples in cache   :        388
Final aggregated edges       :         52
Edges deduped vs Step 1      :         52  (already excluded)

Step 1 unique node pairs     : 14,790,106
Step 2 new unique node pairs :         52
Step 1 unique nodes involved :     26,132
Step 2 unique nodes involved :         92


In [27]:
# Cell 12 — save nlp_kg_edges_llm.parquet
if not llm_edges_df.empty:
    llm_edges_df.to_parquet(OUTPUT_PATH, index=False)
    print(f"Saved {len(llm_edges_df):,} edges to:")
    print(f"  {OUTPUT_PATH}")
    print(f"\nSchema:")
    print(llm_edges_df.dtypes)
else:
    print("WARNING: llm_edges_df is empty — nothing saved.")
    print("Run Cell 8 to process papers first, then re-run Cells 9–12.")

Saved 52 edges to:
  /Users/borng/code/lab_work/neurovlm/experiments/data/nlp_kg/nlp_kg_edges_llm.parquet

Schema:
subject_id           str
relation_type        str
object_id            str
source               str
weight           float64
source_kg            str
dtype: object


In [28]:
# Cell 13 — projected KG distribution after merging Step 1 + Step 2
combined = pd.concat(
    [step1_df[["relation_type"]], llm_edges_df[["relation_type"]]],
    ignore_index=True,
)
combined_counts = combined["relation_type"].value_counts()
combined_total  = len(combined)

print("Projected NLP-KG edge distribution after Step 1 + Step 2 merge:")
print(f"{'relation_type':<30}  {'count':>12}  {'%':>7}")
print("-" * 56)
for rel, cnt in combined_counts.items():
    print(f"{rel:<30}  {cnt:>12,}  {100*cnt/combined_total:>6.1f}%")
print("-" * 56)
print(f"{'TOTAL':<30}  {combined_total:>12,}  {100.0:>6.1f}%")

print(f"\nStep 2 added {len(llm_edges_df):,} net-new edges ({100*len(llm_edges_df)/combined_total:.2f}% of combined pool)")

Projected NLP-KG edge distribution after Step 1 + Step 2 merge:
relation_type                          count        %
--------------------------------------------------------
implicated_in                      6,619,101    44.8%
associated_with_disorder           4,185,358    28.3%
treated_by                         2,695,381    18.2%
used_in                            1,290,297     8.7%
expressed_in                              19     0.0%
co_activates_with                          2     0.0%
--------------------------------------------------------
TOTAL                             14,790,158   100.0%

Step 2 added 52 net-new edges (0.00% of combined pool)
